In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import warnings

# Suppress warnings
warnings.filterwarnings('ignore')

# This script will load each of the 4 datasets, pre-process them,
# and then calculate and show the correlation matrix.
# Since a full matrix is hard to read, we will also show the
# correlation of all features with a *single* target variable, sorted.

# --- List of files and their likely target variables ---
files_and_targets = {
    "merged_weather_data.csv": "Rain (mm)",
    "_crop+yield+prediction_data_crop_yield.csv": "Yield",
    # This one was named in a code block, let's assume it's in the folder
    "telangana_soil_data_2015_2025.csv": "Phosphorus (kg/ha)", 
    "final_crop_yield_dataset.csv": "Rain (mm)" # This file has many targets, let's pick one
}

def preprocess_and_correlate(filename, target_col_name):
    """
    Loads, preprocesses, and prints correlation analysis for a single file.
    """
    print("="*80)
    print(f"ANALYZING FILE: {filename}")
    print("="*80)
    
    try:
        df = pd.read_csv(filename)
    except FileNotFoundError:
        print(f"Error: File '{filename}' not found. Skipping.\n")
        return
    except Exception as e:
        print(f"Error loading {filename}: {e}. Skipping.\n")
        return

    # --- Pre-processing ---
    
    # 1. Handle Dates
    if 'Date' in df.columns:
        try:
            df['Date'] = pd.to_datetime(df['Date'], dayfirst=True, errors='coerce')
            df['Month'] = df['Date'].dt.month
            df['Day'] = df['Date'].dt.day
            df = df.drop(columns=['Date'])
        except Exception as e:
            print(f"  Warning: Could not parse 'Date' in {filename}. Dropping it.")
            df = df.drop(columns=['Date'])

    # 2. Encode All Categorical (object) Columns
    le = LabelEncoder()
    categorical_cols = df.select_dtypes(include=['object']).columns
    if not categorical_cols.empty:
        print(f"  Encoding categorical columns: {list(categorical_cols)}")
        for col in categorical_cols:
            df[col] = le.fit_transform(df[col])
    
    # 3. Handle Missing Values
    df.dropna(inplace=True)
    if df.empty:
        print("  Error: No data left after dropping missing values. Skipping.\n")
        return

    # --- Correlation Analysis ---
    print(f"\n  Calculating correlations...")
    try:
        corr_matrix = df.corr()
        
        # Check if the target column exists (it might have a typo or be case-sensitive)
        # We need to find the *actual* column name, since the user's was a guess
        actual_target_col = None
        if target_col_name in corr_matrix:
            actual_target_col = target_col_name
        else:
            # Check for case-insensitive match
            for col in corr_matrix.columns:
                if col.lower() == target_col_name.lower():
                    actual_target_col = col
                    break
        
        if actual_target_col:
            print("\n  --- Correlation with Target Variable (" + actual_target_col + ") ---")
            # Get correlation of all features with the target, sort descending
            target_corr = corr_matrix[actual_target_col].sort_values(ascending=False)
            print(target_corr)
        else:
            print(f"\n  Warning: Target '{target_col_name}' not found in {filename}.")
            print("  Showing full correlation matrix instead (first 15x15):")
            print(corr_matrix.iloc[:15, :15])
            
    except Exception as e:
        print(f"  Error during correlation: {e}")
        print("  DataFrame info:")
        df.info()

    print("\n" + "="*80 + "\n")


# --- Run the analysis for all 4 files ---
for file, target in files_and_targets.items():
    preprocess_and_correlate(file, target)

print("All file analyses complete.")

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
import warnings

# Suppress warnings
warnings.filterwarnings('ignore')

# --- Define the full file paths you provided ---
PATH_FINAL_DATASET = r"E:\Project\datasets\data\final_crop_yield_dataset.csv"
PATH_CROP_YIELD_DATASET = r"E:\Project\datasets\data\_crop+yield+prediction_data_crop_yield.csv"

# --- Part 1: Analyze the Merged "Master" Dataset ---

print(f"Loading master file: {PATH_FINAL_DATASET}...")
try:
    # We load ONLY the master file, using the full path
    df_final = pd.read_csv(PATH_FINAL_DATASET, low_memory=False)
except FileNotFoundError:
    print(f"Error: File not found at '{PATH_FINAL_DATASET}'")
    print("Please double-check the path and try again. Exiting.")
    # In a real script, we'd exit()
    df_final = None 
except Exception as e:
    print(f"An error occurred loading {PATH_FINAL_DATASET}: {e}")
    df_final = None

if df_final is not None:
    print("Preprocessing df_final...")
    # 1. Handle Missing Data
    df_final.dropna(inplace=True)

    # 2. Handle Date Column
    try:
        if 'Date' in df_final.columns:
            df_final['Date'] = pd.to_datetime(df_final['Date'], errors='coerce')
            df_final.dropna(subset=['Date'], inplace=True) # Drop rows where date failed
            df_final['Month'] = df_final['Date'].dt.month
            df_final['Day'] = df_final['Date'].dt.day
            df_final = df_final.drop(columns=['Date'])
    except Exception as e:
        print(f"Warning processing Date: {e}")

    # 3. Encode Categorical Columns
    le = LabelEncoder()
    categorical_cols = df_final.select_dtypes(include=['object']).columns
    for col in categorical_cols:
        df_final[col] = le.fit_transform(df_final[col])

    print("Calculating correlation matrix for df_final...")
    # Calculate the correlation matrix
    corr_matrix_final = df_final.corr()

    # --- Create and show a heatmap WITH VALUES ---
    print("Displaying heatmap for the final_crop_yield_dataset (with values)...")
    print("NOTE: This may be very dense and hard to read.")
    
    plt.figure(figsize=(24, 22))
    sns.heatmap(
        corr_matrix_final, 
        annot=True,                 # <-- THIS IS THE CHANGE: Show values
        cmap='coolwarm', 
        fmt='.2f',                  # Format values to 2 decimal places
        annot_kws={"size": 8}       # Set the font size for the values
    )
    plt.title('Correlation Heatmap for final_crop_yield_dataset.csv (with values)', fontsize=20)
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show() # Show the plot

    # --- Extracting Interesting Correlations from the big matrix ---
    print("\n--- High & Low Correlations in final_crop_yield_dataset.csv ---")

    # Unstack the matrix to a Series
    corr_series = corr_matrix_final.unstack()
    corr_series = corr_series.drop_duplicates()
    corr_series = corr_series[corr_series != 1] # drop self-correlations

    # Sort the series
    sorted_corr = corr_series.sort_values(ascending=False)

    print("\n** Top 20 MOST POSITIVE Correlations **")
    print(sorted_corr.head(20))

    print("\n** Top 20 MOST NEGATIVE Correlations **")
    print(sorted_corr.tail(20))


# --- Part 2: Analyze the Separate Yield Dataset ---

print(f"\n\nLoading separate file: {PATH_CROP_YIELD_DATASET}...")
try:
    # We load the SECOND file, using the full path
    df_crop_yield = pd.read_csv(PATH_CROP_YIELD_DATASET)
except FileNotFoundError:
    print(f"Error: File not found at '{PATH_CROP_YIELD_DATASET}'")
    print("Please double-check the path.")
    df_crop_yield = None
except Exception as e:
    print(f"An error occurred loading {PATH_CROP_YIELD_DATASET}: {e}")
    df_crop_yield = None

if df_crop_yield is not None:
    # Preprocess: Encode 'Crop'
    le_crop = LabelEncoder()
    df_crop_yield['Crop'] = le_crop.fit_transform(df_crop_yield['Crop'])

    print("Calculating correlation matrix for df_crop_yield...")
    corr_matrix_yield = df_crop_yield.corr()

    print("\n--- Correlations with 'Yield' (from _crop+yield+prediction_data_crop_yield.csv) ---")
    # Get correlations with Yield, then sort
    yield_corr = corr_matrix_yield['Yield'].sort_values(ascending=False)
    print(yield_corr)

print("\nAnalysis complete.")

In [ ]:
# ==============================
# CORRELATION ANALYSIS & SAVE
# ==============================

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
import warnings
import os

# ------------------------------
# Suppress warnings
# ------------------------------
warnings.filterwarnings('ignore')

# ------------------------------
# File Paths
# ------------------------------
PATH_FINAL_DATASET = r"E:\Project\datasets\data\final_crop_yield_dataset.csv"
PATH_CROP_YIELD_DATASET = r"E:\Project\datasets\data\_crop+yield+prediction_data_crop_yield.csv"

# Output directory
OUTPUT_DIR = r"E:\Project"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ------------------------------
# Utility Function
# ------------------------------
def encode_categorical_columns(df):
    le = LabelEncoder()
    for col in df.select_dtypes(include=['object']).columns:
        df[col] = le.fit_transform(df[col])
    return df

# =========================================================
# PART 1: FINAL MERGED DATASET CORRELATION
# =========================================================
print(f"\nLoading master dataset:\n{PATH_FINAL_DATASET}")

try:
    df_final = pd.read_csv(PATH_FINAL_DATASET, low_memory=False)
except Exception as e:
    print("❌ Error loading final dataset:", e)
    df_final = None

if df_final is not None:
    print("Preprocessing final dataset...")

    # Remove missing values
    df_final.dropna(inplace=True)

    # Date processing
    if 'Date' in df_final.columns:
        df_final['Date'] = pd.to_datetime(df_final['Date'], errors='coerce')
        df_final.dropna(subset=['Date'], inplace=True)
        df_final['Month'] = df_final['Date'].dt.month
        df_final['Day'] = df_final['Date'].dt.day
        df_final.drop(columns=['Date'], inplace=True)

    # Encode categorical columns
    df_final = encode_categorical_columns(df_final)

    # Correlation Matrix
    print("Calculating correlation matrix (final dataset)...")
    corr_final = df_final.corr()

    # ------------------------------
    # Save Heatmap
    # ------------------------------
    FINAL_HEATMAP_PATH = os.path.join(
        OUTPUT_DIR, "final_dataset_correlation_heatmap.png"
    )

    plt.figure(figsize=(24, 22))
    sns.heatmap(
        corr_final,
        cmap='coolwarm',
        annot=True,
        fmt='.2f',
        annot_kws={"size": 8}
    )
    plt.title(
        "Correlation Heatmap – Final Crop Yield Dataset",
        fontsize=20
    )
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig(FINAL_HEATMAP_PATH, dpi=300, bbox_inches='tight')
    plt.show()

    print(f"✅ Final dataset heatmap saved at:\n{FINAL_HEATMAP_PATH}")

    # ------------------------------
    # Extract Strong Correlations
    # ------------------------------
    print("\nExtracting strongest correlations...")

    corr_series = corr_final.unstack()
    corr_series = corr_series.drop_duplicates()
    corr_series = corr_series[corr_series != 1.0]
    sorted_corr = corr_series.sort_values(ascending=False)

    print("\n🔹 Top 20 POSITIVE correlations")
    print(sorted_corr.head(20))

    print("\n🔹 Top 20 NEGATIVE correlations")
    print(sorted_corr.tail(20))

    # Save correlations to CSV
    CORR_CSV_PATH = os.path.join(
        OUTPUT_DIR, "final_dataset_top_correlations.csv"
    )
    sorted_corr.to_csv(CORR_CSV_PATH)
    print(f"✅ Correlation values saved at:\n{CORR_CSV_PATH}")

# =========================================================
# PART 2: CROP–YIELD DATASET ANALYSIS
# =========================================================
print(f"\nLoading crop-yield dataset:\n{PATH_CROP_YIELD_DATASET}")

try:
    df_yield = pd.read_csv(PATH_CROP_YIELD_DATASET)
except Exception as e:
    print("❌ Error loading crop-yield dataset:", e)
    df_yield = None

if df_yield is not None:
    print("Preprocessing crop-yield dataset...")

    # Encode Crop column
    if 'Crop' in df_yield.columns:
        le_crop = LabelEncoder()
        df_yield['Crop'] = le_crop.fit_transform(df_yield['Crop'])

    # Correlation Matrix
    print("Calculating correlation matrix (crop-yield dataset)...")
    corr_yield = df_yield.corr()

    # ------------------------------
    # Yield Correlation
    # ------------------------------
    if 'Yield' in corr_yield.columns:
        yield_corr = corr_yield['Yield'].sort_values(ascending=False)
        print("\n🔹 Correlation with Yield")
        print(yield_corr)

        # Save Yield correlation
        YIELD_CSV_PATH = os.path.join(
            OUTPUT_DIR, "yield_feature_correlations.csv"
        )
        yield_corr.to_csv(YIELD_CSV_PATH)
        print(f"✅ Yield correlations saved at:\n{YIELD_CSV_PATH}")

        # ------------------------------
        # Save Yield Bar Plot
        # ------------------------------
        YIELD_PLOT_PATH = os.path.join(
            OUTPUT_DIR, "yield_correlation_barplot.png"
        )

        plt.figure(figsize=(10, 6))
        yield_corr.plot(kind='bar')
        plt.title("Feature-wise Correlation with Yield")
        plt.ylabel("Correlation Value")
        plt.tight_layout()
        plt.savefig(YIELD_PLOT_PATH, dpi=300, bbox_inches='tight')
        plt.show()

        print(f"✅ Yield correlation plot saved at:\n{YIELD_PLOT_PATH}")

print("\n🎯 CORRELATION ANALYSIS COMPLETED SUCCESSFULLY")